# 01 — Acquisition and data quality

**Question this notebook answers:** can the 569 rows be trusted, and does anything in
them need repairing before modelling?

The dataset is the Wisconsin Diagnostic Breast Cancer (WDBC) set: 30 numeric features
computed from digitised images of fine-needle aspirates of breast masses, each labelled
benign or malignant. Every feature describes the *cell nuclei* visible in the image —
their size, their texture, the irregularity of their outline.

Two things get settled here. First, the data is acquired by code rather than by hand, so
the provenance is checkable. Second, there is exactly one data-quality question in this
dataset worth arguing about, and it has a non-obvious answer.

## Why download from UCI rather than import from scikit-learn

`sklearn.datasets.load_breast_cancer()` returns this exact data in one line. Using it
would have been faster and it is what most published versions of this project do. Two
reasons not to:

1. **The bundled copy drops the `id` column.** Without patient IDs there is no way to
   check whether 569 rows are 569 *patients*. If any patient appeared twice, rows in a
   cross-validation training fold would be duplicated in its validation fold, and every
   performance number after that would be inflated. The check is cheap; skipping it
   means reporting a number you cannot defend.
2. **`wdbc.names` carries the measurement protocol** and the original authors' own
   result. That result turns out to set the benchmark this project has to beat
   (notebook 03), and it is not shipped with the scikit-learn copy.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

from src import data as D
from src import profiling as P

pd.set_option("display.width", 160, "display.max_columns", 40)


In [2]:
# Idempotent: skips the network call if the archive is already present.
from src.acquisition import main as acquire
frame = acquire()

Archive already present, skipping download: wdbc.zip
Already extracted: wdbc.data
Already extracted: wdbc.names
Provenance written to provenance.json

569 rows x 32 columns -> wdbc_raw.parquet
Class balance: {'B': 357, 'M': 212}
SHA-256 (wdbc.data): d606af411f3e5be8a317a5a8b652b425aaf0ff38ca683d5327ffff94c3695f4a


The provenance record written alongside the data pins the SHA-256 of the exact
bytes used. If UCI ever revises the file, the checksum changes and the discrepancy is
visible rather than silent.

In [3]:
import json
print(json.dumps(json.loads((D.PROJECT_ROOT / "data/raw/provenance.json").read_text()), indent=2)[:900])

{
  "source_url": "https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip",
  "citation": "Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). Breast Cancer Wisconsin (Diagnostic) [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B",
  "licence": "CC BY 4.0",
  "retrieved_utc": "2026-07-22T15:42:56+00:00",
  "archive_filename": "wdbc.zip",
  "archive_sha256": "bc154869ef13f753f9e2b5a17e248cfe1ba4b6721db7c4da9f4880e40b05d3af",
  "archive_bytes": 51284,
  "members": {
    "data": {
      "filename": "wdbc.data",
      "sha256": "d606af411f3e5be8a317a5a8b652b425aaf0ff38ca683d5327ffff94c3695f4a",
      "bytes": 124103
    },
    "names": {
      "filename": "wdbc.names",
      "sha256": "840e04e3f20f8a5b326892f3b9cbc01c4cd6f7e6c597630b701ef6c0ac79f5ef",
      "bytes": 4708
    }
  },
  "rows": 569,
  "columns": 32,
  "class_coun


**Licence.** CC BY 4.0 — redistribution is permitted with attribution. The data is
still not committed to the repository: downloading at runtime keeps the provenance
record and the file guaranteed consistent, and keeps the repo small. The citation is in
`src/acquisition.py` and the README.

## The structure of the 30 features

Ten measurements are taken of each cell nucleus. Each is then summarised three ways
across the nuclei in one image — the **mean**, the **standard error**, and the
**"worst"** value (the mean of the three largest). Ten measurements times three
summaries gives the 30 columns.

This matters more than it first appears. The columns are not 30 independent facts about
a tumour; they are 10 facts recorded at three levels of aggregation. Notebook 02 shows
what that does to the correlation structure.

In [4]:
print("Base measurements:", ", ".join(D.BASE_FEATURES))
print()
frame.head(3).iloc[:, :8]

Base measurements: radius, texture, perimeter, area, smoothness, compactness, concavity, concave_points, symmetry, fractal_dimension



,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990


## Data quality

The checks that matter before any modelling: missing values, duplicate rows, duplicate
patients, and the class balance the split will have to preserve.

In [5]:
quality = P.quality_report(frame)
for key, value in quality.items():
    print(f"{key:34s} {value}")

rows                               569
features                           30
missing_values                     0
duplicate_rows                     0
duplicate_ids                      0
duplicate_feature_rows             0
class_counts                       {'B': 357, 'M': 212}
malignant_rate                     0.37258347978910367
zero_concavity_rows                13
zero_concavity_is_all_or_nothing   True
zero_concavity_diagnoses           {'B': 13}
scale_ratio                        215170.71905908443


Clean on every conventional axis: **no missing values, no duplicate rows, and 569
distinct patient IDs**, so the independence assumption behind cross-validation holds.

Two things in that report do need acting on.

**The classes are 37% malignant / 63% benign.** Not severe imbalance, but enough that
an unstratified 20% test draw would let the malignant share wander by several points —
and at n=114 that moves every metric reported against it. Every split and every CV fold
in this project is stratified.

**Feature scales span a factor of ~215,000.** `area_worst` has a standard deviation of
about 569; `fractal_dimension_se` about 0.0026. Any penalised or distance-based model
fitted to raw columns would be fitted almost entirely to the area measurements.
Standardisation is not optional here, and it has to happen *inside* the
cross-validation pipeline — see notebook 03.

## The one real data-quality question: 13 rows of zeros

Six columns contain exact zeros: the concavity and concave-points measurements, at all
three aggregation levels. Zeros in a continuous measurement column are the classic
signature of a missing-value code, and the reflex is to impute them.

Before doing that, two things are worth establishing.

In [6]:
zeros = (frame[D.ZERO_CONCAVITY_COLUMNS] == 0)
print("rows zero on ALL six concavity columns:", zeros.all(axis=1).sum())
print("rows zero on ANY of the six:          ", zeros.any(axis=1).sum())
print("=> the same rows:", bool((zeros.all(axis=1) == zeros.any(axis=1)).all()))

rows zero on ALL six concavity columns: 13
rows zero on ANY of the six:           13
=> the same rows: True


**All-or-nothing.** No row is zero on some concavity columns but not others. A
sporadic missing-value code would scatter; this does not. Whatever produced these zeros
acted on the whole concavity family of one image at once.

That is consistent with a real measurement. `concavity` measures the severity of concave
portions of the nucleus outline, and `concave_points` counts them. A **perfectly convex
contour has none** — so all six columns are legitimately zero together, and the standard
error is zero because every nucleus in the image agrees.

If that reading is right, these should be small, round nuclei — which means small
tumours.

In [7]:
P.zero_concavity_evidence(frame).round(3)

,zero_concavity,all_others,benign_others,zero_conc_mean_percentile
radius_mean,9.411,14.238,12.250,0.078
area_mean,276.569,663.735,469.828,0.079
perimeter_mean,59.138,92.737,78.791,0.070


In [8]:
zero_rows = zeros.all(axis=1)
print(frame.loc[zero_rows, "diagnosis"].value_counts().to_string())
print()
print(frame.loc[zero_rows, ["id", "radius_mean", "area_mean",
                            "compactness_mean", "concavity_mean", "diagnosis"]]
      .to_string(index=False))

diagnosis
B    13

     id  radius_mean  area_mean  compactness_mean  concavity_mean diagnosis
 862722        6.981      143.5           0.07568             0.0         B
 868999        9.738      288.5           0.04102             0.0         B
 871642       10.660      349.6           0.04302             0.0         B
 872113        8.671      227.2           0.04276             0.0         B
 875099        9.720      288.1           0.02344             0.0         B
 894047        8.597      221.2           0.05847             0.0         B
 903483        8.734      234.3           0.07428             0.0         B
9113846       12.270      465.4           0.03398             0.0         B
 921092        7.729      178.8           0.04878             0.0         B
 923748       10.860      360.5           0.04227             0.0         B
 925236        9.423      271.3           0.04971             0.0         B
 925311       11.200      386.0           0.03558             0.0    

**Verdict: these are real measurements, and imputing them would destroy signal.**

The evidence:

- They sit in the **bottom ~8th percentile** of every size measurement. Their mean radius
  is 9.41 against 14.24 for everything else — one of them, at radius 6.98, is the
  smallest mass in the dataset.
- **All 13 are benign.** Not 12, not 11. Zero concavity is perfectly predictive of
  benignity in this sample.
- `compactness` — computed as `perimeter² / area − 1`, mathematically unrelated to
  concavity — is present and non-zero for all 13. A truncated or failed image-processing
  step would not have selectively spared it.

So the zeros are a **biological floor, not a missing-value code**: small benign masses
have round, convex nuclei with nothing to measure. Replacing them with a column median
would push 13 benign cases toward the malignant region of feature space and delete the
strongest categorical signal in the dataset.

The one caveat to hold onto: 13/13 benign is a small sample. A model can lean on
`concavity == 0` more confidently than 13 patients justify, and this is exactly the kind
of rule that would not survive to a new cohort. It is not corrected for here — it is
recorded, and revisited in the limitations of notebook 03.

`tests/test_data.py::test_zero_concavity_rows_are_all_or_nothing` pins this so the claim
stays checkable.

## What carries forward

| Finding | Consequence |
|---|---|
| 569 unique patients, no missing values, no duplicates | No cleaning step required; CV folds are independent |
| 37% malignant | Every split stratified |
| Feature scales span ~215,000x | Standardise, inside the CV pipeline |
| 13 zero-concavity rows are real, all benign | **No imputation**; flagged as a small-sample risk |
| 30 columns = 10 measurements x 3 aggregations | Expect heavy collinearity — quantified in notebook 02 |

Next: **02_eda.ipynb** — what the features actually say about the classes, and how much
independent information there is among 30 columns.